# Quantum Fourier Transform

The QFT maps $|j\rangle$ to
$$
\mathrm{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i\, jk/N}|k\rangle
$$

It is the quantum analogue of the discrete Fourier transform and the
core subroutine in Shor's algorithm and quantum phase estimation.

This notebook builds the QFT gate-by-gate on 3–4 qubits, verifies
against the classical DFT matrix, and demos frequency decomposition.

In [ ]:
import cmath, math
import qiskit as qk
import qiskit_aer as qka

## QFT circuit (Qiskit little-endian)

For qubit $j$ (LSB = qubit 0): apply $H$, then controlled-$R_k$
rotations from qubits $0,\dots,j-1$, then SWAP to reverse bit order.

In [ ]:
def qft_circuit(n):
    qc = qk.QuantumCircuit(n, name=f"QFT-{n}")
    for j in range(n):
        qc.h(j)
        for k in range(j):
            qc.cp(math.pi / 2 ** (j - k), k, j)
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    return qc

def inverse_qft_circuit(n):
    qc = qk.QuantumCircuit(n, name=f"IQFT-{n}")
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    for j in range(n - 1, -1, -1):
        for k in range(j - 1, -1, -1):
            qc.cp(-math.pi / 2 ** (j - k), k, j)
        qc.h(j)
    return qc

print(qft_circuit(3).draw())

## Verify against classical DFT

In [ ]:
def dft_matrix(n):
    N = 2**n
    return [[cmath.exp(2j * math.pi * j * k / N) / math.sqrt(N) for k in range(N)] for j in range(N)]

for n in (3, 4):
    qc = qk.QuantumCircuit(n)
    qc.compose(qft_circuit(n), inplace=True)
    unitary = qk.quantum_info.Operator(qc).data
    dft = dft_matrix(n)
    max_err = max(abs(unitary[j][k] - dft[j][k]) for j in range(2**n) for k in range(2**n))
    print(f"n={n}: max error vs DFT matrix = {max_err:.2e}")

## Frequency decomposition

Apply QFT to $|2\rangle$ on 3 qubits. The output distributes amplitude
across all basis states with phase factors $e^{2\pi i \cdot 2k/8}$.

In [ ]:
n = 3
j_val = 2
qc = qk.QuantumCircuit(n)
for bit in range(n):
    if (j_val >> bit) & 1:
        qc.x(bit)
qc.compose(qft_circuit(n), inplace=True)
sv = qk.quantum_info.Statevector.from_instruction(qc)
for state in sorted(sv.probabilities_dict().keys()):
    k = int(state[::-1], 2)
    print(f"  |{state}> (k={k})  p={sv.probabilities_dict()[state]:.4f}  amp={sv.data[k]:.4f}")

## Uniform superposition -> |0>

QFT of $|+\rangle^{\otimes n}$ should collapse to $|0\rangle^{\otimes n}$.

In [ ]:
n = 3
qc = qk.QuantumCircuit(n)
qc.h(range(n))
sv = qk.quantum_info.Statevector.from_instruction(qc)
qc2 = qk.QuantumCircuit(n)
qc2.compose(qft_circuit(n), inplace=True)
sv_out = sv.evolve(qc2)
print("QFT(|+>^3) probabilities:")
for s, p in sorted(sv_out.probabilities_dict().items()):
    if p > 0.01:
        print(f"  |{s}>  p={p:.4f}")

## QFT -> IQFT roundtrip

In [ ]:
n = 3
j_val = 3
qc = qk.QuantumCircuit(n)
for bit in range(n):
    if (j_val >> bit) & 1:
        qc.x(bit)
qc.compose(qft_circuit(n), inplace=True)
qc.compose(inverse_qft_circuit(n), inplace=True)
sv = qk.quantum_info.Statevector.from_instruction(qc)
print(f"roundtrip |{j_val}> -> QFT -> IQFT:")
for s, p in sorted(sv.probabilities_dict().items()):
    if p > 0.01:
        print(f"  |{s}>  p={p:.4f}")

## Sampling QFT|2>

In [ ]:
n = 3
qc = qk.QuantumCircuit(n, n)
qc.x(1)
qc.compose(qft_circuit(n), inplace=True)
qc.measure(range(n), range(n))
print(qc.draw())

sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc, sim), shots=4096).result().get_counts()
for bits, cnt in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  |{bits}>  {cnt:4d}")